In [1]:
import os
os.chdir('/Users/jingyuan/Documents/ChatGPT/模型构建/reproduction/runs/run_02_2026-09-23/results/original_notebook_outputs')

In [2]:
import pandas as pd
import numpy as np
import esm
import torch

print(pd.__version__)
print(np.__version__)
print(esm.__version__)
print(torch.__version__)

2.2.3
1.26.4
2.0.0
2.6.0


In [3]:
import numpy as np
import pandas as pd

In [4]:
import pandas as pd
dataset=pd.read_excel('/Users/jingyuan/Documents/ChatGPT/模型构建/skinaging_predictor_ZJU/Database/Oringinal_data.xlsx', na_filter=False)
sequence_list=dataset['Sequence']

In [5]:
print(sequence_list)

0          GEKG
1         KTTKS
2            AH
3        EEMQRR
4        FVAPFP
         ...   
415    RPKHPIKH
416          QK
417          YP
418        AIPP
419      DVITGA
Name: Sequence, Length: 420, dtype: object


In [6]:
peptide_sequence_list = []
for seq in sequence_list:
    format_seq = [seq,seq]
    tuple_sequence = tuple(format_seq)
    peptide_sequence_list.append(tuple_sequence)
#print(peptide_sequence_list)

In [7]:
#调试完毕，以此为准！
import torch
import esm
import pandas as pd

#load ESM-2 model
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval()
    
#load data
data = peptide_sequence_list
batch_labels, batch_strs, batch_tokens = batch_converter(data)
batch_lens = (batch_tokens != alphabet.padding_idx).sum(1)
    
# Extract per-residue representations (on CPU)
with torch.no_grad():
    results = model(batch_tokens, repr_layers=[6], return_contacts=True)
token_representations = results["representations"][6]
    
# Generate per-sequence representations via averaging
sequence_representations = []
for i, token_len in enumerate(batch_lens):
    each_seq_rep = token_representations[i, 1:token_len - 1].mean(0).tolist()  
    sequence_representations.append(each_seq_rep)

embedding_results = pd.DataFrame(sequence_representations)
print(embedding_results)
embedding_results.to_csv('output_name.csv')

          0         1         2         3         4         5         6    \
0    0.139993 -0.435623  0.533433  0.245520 -0.037778 -0.205565 -0.174131   
1   -0.017159 -0.274750  0.139635  0.226395  0.091278 -0.054094 -0.197419   
2    0.114226 -0.307165  0.206606  0.071511 -0.145882 -0.073673 -0.435139   
3    0.081766 -0.434443  0.517479  0.393443 -0.139181 -0.236885 -0.208077   
4   -0.033665 -0.356248  0.333965  0.183566  0.087931  0.052072 -0.067219   
..        ...       ...       ...       ...       ...       ...       ...   
415 -0.003021 -0.453516  0.212517  0.372837 -0.076083 -0.280915 -0.262101   
416  0.130265 -0.308018  0.533882  0.237206 -0.144475  0.031353 -0.250038   
417  0.080212 -0.378808  0.331284  0.324174  0.134444  0.039193 -0.230765   
418  0.023382 -0.347926  0.248083  0.179577  0.179818  0.002932 -0.144716   
419  0.252434 -0.100778  0.357205  0.011527  0.028443 -0.050235 -0.124899   

          7         8         9    ...       310       311       312  \
0  